In [ ]:
#imports
import nibabel as nib
#import nilearn
import numpy as np
import os
import cv2

import cc3d
from matplotlib import pyplot as plt

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

#from tqdm import tqdm
from skimage.io import imread, imshow
from skimage.transform import resize

In [ ]:
import tensorflow as tf

import keras.backend as K
from keras.models import Model, load_model
from keras.layers import Input, BatchNormalization, Activation, Dense, Dropout
from keras.layers.core import Lambda, RepeatVector, Reshape
from keras.layers.convolutional import Conv2D, Conv2DTranspose
from keras.layers.pooling import MaxPooling2D, GlobalMaxPool2D
from tensorflow.keras.layers import concatenate
#from keras.layers.merge import concatenate, add
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.callbacks import CSVLogger
#from keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, TensorBoard
from tensorflow.keras.layers.experimental import preprocessing

from tensorflow.keras.losses import binary_crossentropy

#from focal_loss import BinaryFocalLoss

In [ ]:
# defining the performance metrics
def dice_coeff(y_true,y_pred):
    y_true_new = K.flatten(y_true)
    y_pred_new = K.flatten(y_pred)
    denominator = K.sum(y_true_new) + K.sum(y_pred_new)
    if denominator == 0.0:
        return 1.0
    numerator = K.sum(y_true_new * y_pred_new)
    return (2.0*numerator + 0.0002)/(denominator + 0.0002)

def dice_coeff_loss(y_true,y_pred):
    return 1.0- dice_coeff(y_true,y_pred)

def iou(y_true,y_pred):
    intersec = K.sum(y_true * y_pred)
    union = K.sum(y_true + y_pred)
    if union == 0.0:
        return 1.0
    iou = (intersec) / (union- intersec)
    return iou

# def dice_score(y_true, y_pred):
#     intersection = tf.reduce_sum(y_true * y_pred)
#     union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
#     dice = (2.0 * intersection + 1e-5) / (union + 1e-5)
#     return dice

# def dice_loss(y_true, y_pred):
#     return 1.0 -dice_score(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    loss = tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_coeff_loss(y_true, y_pred)
    return loss

In [ ]:
def precision_score_(y_true, y_pred):
    y_true_new = K.flatten(y_true)
    y_pred_new = K.flatten(y_pred)
    intersect = K.sum(y_pred_new*y_true_new)
    total_pixel_pred = K.sum(y_pred_new)
    return intersect/total_pixel_pred

def recall_score_(y_true, y_pred):
    y_true_new = K.flatten(y_true)
    y_pred_new = K.flatten(y_pred)
    intersect = K.sum(y_pred_new*y_true_new)
    total_pixel_truth = K.sum(y_true_new)
    return intersect/total_pixel_truth